In [8]:
import sys
!{sys.executable} -m pip install numpy h5py torch


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [9]:
import time
import numpy as np
import torch
import h5py

from models.cmlp import cMLP, train_model_ista


# ============================================================
# 0. 설정
# ============================================================
USE_DEVICE = "mps"
# "cuda:6" -> GPU 서버 cuda:6
# "cpu"    -> 맥미니 CPU
# "mps"    -> Apple Silicon GPU

h5_path = "simulated/260420_data/three_nodes/signal.h5"

save_path = "3node_lag2_lambda_sweep_binary_scores.npy"
lambda_save_path = "3node_lag2_lambdas.npy"

test_start = 1704
test_end = 2000

lag = 2
hidden = [100]
lr = 1e-2
max_iter = 2000
check_every = 100
activation = "relu"

# lambda sweep: 1e-4 ~ 1까지 로그 간격 10개
lambdas = np.logspace(-4, 0, 10)


# ============================================================
# 1. 디바이스 선택
# ============================================================
def get_device(device_name: str):
    if device_name.startswith("cuda"):
        if torch.cuda.is_available():
            return torch.device(device_name)
        print("CUDA is not available. Falling back to CPU.")
        return torch.device("cpu")

    if device_name == "mps":
        if torch.backends.mps.is_available():
            return torch.device("mps")
        print("MPS is not available. Falling back to CPU.")
        return torch.device("cpu")

    return torch.device("cpu")


device = get_device(USE_DEVICE)
print(f"Using device: {device}")


# ============================================================
# 2. 데이터 로드
# ============================================================
with h5py.File(h5_path, "r") as f:
    X_np = f["signal"][()].astype(np.float32)
    # 원본: (simulation, feature, timestep) = (2000, 3, 2000)

X_np = X_np.transpose(0, 2, 1)
# 변환: (simulation, timestep, feature) = (2000, 2000, 3)

X_test_np = X_np[:, test_start:test_end, :]
# test: (simulation, test_timestep, feature) = (2000, 296, 3)

print(f"Loaded X_np shape: {X_np.shape}")
print(f"Test X shape: {X_test_np.shape}")
print(f"Lambdas: {lambdas}")


# ============================================================
# 3. simulation 하나 + lambda 하나 실행
# ============================================================
def run_single_simulation_with_lambda(X_single_np, lam):
    """
    X_single_np: (test_timestep, feature)
    return: binary GC matrix, shape (feature, feature)
    """

    X_single = torch.tensor(
        X_single_np,
        dtype=torch.float32
    ).unsqueeze(0).to(device)
    # (1, test_timestep, feature)

    p = X_single.shape[-1]

    cmlp = cMLP(
        num_series=p,
        lag=lag,
        hidden=hidden,
        activation=activation,
    ).to(device)

    train_model_ista(
        cmlp,
        X_single,
        lam=float(lam),
        lr=lr,
        max_iter=max_iter,
        check_every=check_every,
        verbose=0,
    )

    # lambda에 의해 0이 된 weight group을 기준으로 binary GC 산출
    gc_binary = cmlp.GC().detach().cpu().numpy()
    # (p, p)

    return gc_binary


# ============================================================
# 4. lambda sweep + 전체 simulation 반복
# ============================================================
num_lambdas = len(lambdas)
num_simulations = X_test_np.shape[0]
p = X_test_np.shape[-1]

results = np.zeros((num_lambdas, num_simulations, p, p), dtype=np.float32)
# 결과: (lambda, simulation, row, column)
# 예: (10, 2000, 3, 3)

np.save(lambda_save_path, lambdas)

start_time = time.time()

for lam_idx, lam in enumerate(lambdas):
    lambda_start = time.time()

    print(f"\n========== Lambda {lam_idx + 1}/{num_lambdas}: {lam:.6g} ==========")

    for sim_idx in range(num_simulations):
        sim_start = time.time()

        results[lam_idx, sim_idx] = run_single_simulation_with_lambda(
            X_test_np[sim_idx],
            lam=lam,
        )

        sim_elapsed = time.time() - sim_start

        if (sim_idx + 1) % 10 == 0 or sim_idx == 0:
            total_elapsed = time.time() - start_time
            print(
                f"[lambda {lam_idx + 1}/{num_lambdas} | "
                f"sim {sim_idx + 1}/{num_simulations}] "
                f"sim_time={sim_elapsed:.2f}s, "
                f"total={total_elapsed / 60:.2f}min"
            )

            # 중간 저장
            np.save(save_path, results)

    lambda_elapsed = time.time() - lambda_start
    print(
        f"Lambda {lam:.6g} done: "
        f"{lambda_elapsed:.2f}s ({lambda_elapsed / 60:.2f}min)"
    )

    np.save(save_path, results)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"\nTotal time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} min)")
print(f"Final results shape: {results.shape}")

np.save(save_path, results)
np.save(lambda_save_path, lambdas)

print(f"Saved results to: {save_path}")
print(f"Saved lambdas to: {lambda_save_path}")

Using device: mps
Loaded X_np shape: (2000, 2000, 3)
Test X shape: (2000, 296, 3)
Lambdas: [1.00000000e-04 2.78255940e-04 7.74263683e-04 2.15443469e-03
 5.99484250e-03 1.66810054e-02 4.64158883e-02 1.29154967e-01
 3.59381366e-01 1.00000000e+00]

========== Lambda 1/10: 0.0001 ==========
[lambda 1/10 | sim 1/2000] sim_time=5.53s, total=0.09min
[lambda 1/10 | sim 10/2000] sim_time=3.47s, total=0.60min
[lambda 1/10 | sim 20/2000] sim_time=3.32s, total=1.15min
[lambda 1/10 | sim 30/2000] sim_time=3.32s, total=1.72min
[lambda 1/10 | sim 40/2000] sim_time=3.40s, total=2.29min
[lambda 1/10 | sim 50/2000] sim_time=3.31s, total=2.85min
[lambda 1/10 | sim 60/2000] sim_time=3.22s, total=3.41min
[lambda 1/10 | sim 70/2000] sim_time=3.35s, total=3.97min
[lambda 1/10 | sim 80/2000] sim_time=3.48s, total=4.54min
[lambda 1/10 | sim 90/2000] sim_time=3.31s, total=5.09min
[lambda 1/10 | sim 100/2000] sim_time=3.32s, total=5.65min
[lambda 1/10 | sim 110/2000] sim_time=3.31s, total=6.20min
[lambda 1/10 | 

KeyboardInterrupt: 